[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JulesMalin/isba2411-nlp/blob/main/Midterm/ISBA2411_Midterm_Starter.ipynb)

# ISBA 2411 — Midterm Mini-Project
## Review Triage: how bad is this review?

**300 points — worth 30% of your course grade · take-home · budget about 2 hours**

### The situation
You are back on the Review Assistant from Lecture 11. Before the system can draft a reply, it has to
know **how bad a review is** so it can route the urgent ones to a human. Your job: predict a review's
**star rating (1–5) from its text alone**, then report honestly on how well that actually works.

Everyone uses the **same provided data and the same split**, so results are directly comparable.

### Who you work with
This midterm is tied to your **final-project team**. You may **work with your project team** — talk through
the approach, debug together, compare results — but **each person submits their own files**: your own
notebook and your own write-up, in your own words. Identical, copy-pasted submissions are not independent work.

### Rules
- Use only methods from **Weeks 1–6**: tokenization/features, classification, embeddings, retrieval, clustering/topics.
- Pretrained models (TF-IDF, scikit-learn, sentence-embedding models) are **tools you may call**. Do **not** fine-tune/train a transformer, and do not use an LLM to produce the predictions.
- **AI assistants are allowed**, but you must disclose where and how you used them in Part 5. Note that Parts 3–5 ask about *your own* numbers and *your own* misclassified rows — an AI cannot know those.
- Do not swap in outside data. Do not train on the test set.

### What to submit — upload to Camino
1. **A PDF of your completed notebook**, run top to bottom. (In Colab: `File -> Print -> Save as PDF`.)
2. **A link to your Colab notebook** — `Share -> General access -> Anyone with the link -> Viewer`, then copy and submit the link.
3. **Your write-up** — either filled into **Part 5 of this notebook**, or as a **separate document** (PDF or Word) uploaded to Camino alongside the notebook.

### How it is graded (300 pts)
| Criterion | Pts | Where it is earned |
|---|---|---|
| **Execution** | 150 | Parts 1, 2, 4 — correct, working models and metrics |
| **Analysis & Insight** | 100 | Parts 0, 3, 4 — what the numbers *mean* and where the method fails |
| **Communication** | 50 | Part 5 / your write-up — clear and organized |

> **Strategy note.** The single biggest mistake you can make is reporting accuracy and stopping.
> Points live in explaining *why* a number is what it is.

## Setup

Run this first. It loads the frozen train/test split (no downloads to configure).

In [ ]:
%pip install -q scikit-learn sentence-transformers pandas matplotlib

import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

RAW = "https://raw.githubusercontent.com/JulesMalin/isba2411-nlp/main/Midterm"
train = pd.read_csv(f"{RAW}/midterm_reviews_train.csv")
test  = pd.read_csv(f"{RAW}/midterm_reviews_test.csv")
print(train.shape, test.shape)
train.head()

A small helper so every model is reported the same way. Use it for each model you build.

In [ ]:
def score(name, y_true, y_pred):
    """Print the two headline numbers plus the per-class breakdown."""
    acc = accuracy_score(y_true, y_pred)
    f1m = f1_score(y_true, y_pred, average="macro")
    print(f"{name:34} accuracy {acc:.3f} | macro-F1 {f1m:.3f}")
    return {"model": name, "accuracy": round(acc, 3), "macro_f1": round(f1m, 3)}

results = []      # append each score(...) call here so you can build one comparison table

### How to fill in the code cells

You do **not** write code from scratch here. Every code cell is *almost complete* — wherever you see
`____` (four underscores), replace it with the one missing piece. The comment right beside it tells you
exactly what goes there. Everything else is written for you; just run each cell top to bottom.

- If a cell errors with `NameError: name '____' is not defined`, you left a blank unfilled — go back and fill it.
- The real work — and most of the points — is in the **"Your answer"** cells. Those are yours to write, in words.

## Part 0 · Establish the floor  *(~10 min)*

Before any model, find out what "good" even means on this data.

**Do:**
1. Print the star distribution in `train` (counts and percentages).
2. Build the **dumbest possible baseline**: always predict the single most common star rating.
3. Score it with `score(...)`.

**Answer in the markdown cell below:** what accuracy does the do-nothing baseline get, and what does
that tell you about using accuracy as your headline metric here?

In [ ]:
# 1) How common is each star rating in the training data? (proportions, 0-1)
print(train.star.value_counts(normalize=True).sort_index().round(3))

# 2) The dumbest possible model: always guess the single most common rating.
maj = train.star.mode()[0]                 # the most common star in train
baseline_pred = [maj] * len(test)          # predict `maj` for EVERY test review

# 3) Score that baseline. (score() also saves it to `results` for the final table.)
results.append(score("majority (always guesses the top rating)", test.star, ____))   # <- fill in: the baseline predictions

**Your answer (Part 0):**

*(replace this text)*

## Part 1 · Baseline model — TF-IDF  *(~25 min)*

**Do:**
1. Vectorize `review` with `TfidfVectorizer` (your choice of settings — say what you chose).
2. Train a classifier (`LogisticRegression` is fine) to predict `star`.
3. `score(...)` it, then print `classification_report` to see **per-class** results.

**Answer below:** how much did you beat the Part 0 floor by? Is any class doing badly?

In [ ]:
# 1) Turn the review TEXT into numbers with TF-IDF.
vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
Xtr = vec.fit_transform(train.review)      # learn the vocabulary on train, then transform train
Xte = vec.transform(test.____)             # <- fill in: which column holds the review text?

# 2) Train a classifier to predict the star rating from those numbers.
clf = LogisticRegression(max_iter=1000).fit(Xtr, train.star)
pred_tfidf = clf.predict(Xte)              # the model's guesses for the test reviews

# 3) Score it, then print the PER-CLASS breakdown to see which stars it handles well.
results.append(score("TF-IDF + LogReg", test.star, ____))          # <- fill in: the predictions you just made
print(classification_report(test.star, pred_tfidf, zero_division=0))

**Your answer (Part 1):**

*(replace this text)*

## Part 2 · A second representation — embeddings  *(~30 min)*

Swap the *representation*, keep everything else the same, so the comparison is fair.

**Do:**
1. Encode the reviews with a pretrained sentence-embedding model, e.g.
   `SentenceTransformer("all-MiniLM-L6-v2")` (encoding ~8k short reviews takes a couple of minutes on CPU).
2. Train the **same** classifier on those embeddings.
3. `score(...)` it and compare against Part 1.

**Answer below:** did embeddings beat TF-IDF? **Whatever you found, explain why.** Do not assume the
newer method must win — argue from what these reviews actually look like.

In [ ]:
from sentence_transformers import SentenceTransformer

# 1) Load a pretrained embedding model (a "tool you may call", from Lecture 11).
enc = SentenceTransformer("all-MiniLM-L6-v2")

# 2) Turn each review into an embedding vector. (~1-2 min on CPU — this is the slow cell.)
Etr = enc.encode(train.review.tolist(), show_progress_bar=True)
Ete = enc.encode(test.review.tolist(),  show_progress_bar=True)

# 3) Train the SAME kind of classifier — only the representation changed, so it's a fair comparison.
emb_clf = LogisticRegression(max_iter=2000).fit(Etr, train.star)
pred_emb = emb_clf.predict(____)           # <- fill in: the TEST embeddings (Ete), not the training ones
results.append(score("MiniLM embeddings + LogReg", test.star, pred_emb))

**Your answer (Part 2):**

*(replace this text)*

## Part 3 · Diagnose the failure  *(~30 min)*  — the most valuable section

Pick your better model from Parts 1–2 and find out **where and why** it breaks.

**Do:**
1. Plot the **confusion matrix** on the test set (`ConfusionMatrixDisplay.from_predictions`).
2. Report **per-class F1**. Name the class the model handles *worst*.
3. Pull **3 specific misclassified reviews** — print the text, the true star, and the predicted star.

**Answer below, concretely:**
- Which classes get confused with which? Is the confusion **random**, or does it have a pattern?
- Looking at your 3 examples, why is this genuinely hard? (Consider: how long are these reviews? how many
  examples of each class exist? would *you* reliably tell a 2★ from a 3★ from the text alone?)

In [ ]:
# We diagnose the TF-IDF model here (pred_tfidf). If you prefer your embedding model, swap in pred_emb.

# 1) Confusion matrix: which TRUE stars (rows) get predicted as which (columns)?
ConfusionMatrixDisplay.from_predictions(test.star, pred_tfidf, cmap="Blues", colorbar=False)
plt.title("Confusion matrix - true star vs predicted star"); plt.show()

# 2) Per-class F1 again - name the star rating with the LOWEST f1-score.
print(classification_report(test.star, pred_tfidf, zero_division=0))

# 3) Read 3 reviews the model got WRONG (true star is not the predicted star).
errors = test.assign(pred=pred_tfidf).query("star != pred")
for _, row in errors.sample(3, random_state=2411).iterrows():
    print("true =", row.star, " | predicted =", row.____)     # <- fill in: the predicted-star column you made above ("pred")
    print(row.review[:200])
    print()

**Your answer (Part 3):**

*(replace this text)*

## Part 4 · One targeted improvement  *(~20 min)*

Choose **one** change and measure it. Do not try everything.

Options (pick one):
- `class_weight="balanced"` in the classifier
- Collapse the problem to 3 classes (negative 1–2 / neutral 3 / positive 4–5) and re-evaluate
- Tune the TF-IDF features (n-grams, `min_df`, `sublinear_tf`)
- Add a simple hand-built feature (e.g. review length, punctuation) alongside your representation

**Do:** implement it, `score(...)` it, and put all your models in one comparison table.

**Answer below:** did accuracy and macro-F1 move in the **same** direction? If they moved in opposite
directions, which model would you actually ship for the triage system, and why?

In [ ]:
# Pick ONE improvement. This example uses class_weight="balanced", which tells the
# classifier to stop ignoring the rare star ratings. (You only need one change.)
improved = LogisticRegression(max_iter=2000, class_weight="____").fit(Etr, train.star)   # <- fill in: "balanced"
results.append(score("MiniLM + balanced class weights", test.star, improved.predict(Ete)))

# Put EVERY model you built into one comparison table. `results` is the list score() filled in.
import pandas as pd
pd.DataFrame(____)                         # <- fill in: results

**Your answer (Part 4):**

*(replace this text)*

## Part 5 · Write-up  *(~15 min)*

Replace the text below. Keep it under one page.

---

### 1. What I did
*(2–3 sentences: representations tried, classifier, what you changed in Part 4.)*

### 2. Results
*(paste your comparison table)*

### 3. One concrete failure
*(name the specific class or case that fails, with an example, and say why it fails.)*

### 4. Recommendation for the triage system
*(Would you deploy this? For what decision? What would you do instead, or what would you need next?
One honest paragraph — "not good enough to auto-route, but useful for X" is a perfectly good answer
if you support it.)*

### 5. AI use disclosure
*(Required. Which tools, for what — debugging, wording, explaining a concept — or "none.")*